

# 📔 **Resolución**
Una vez que la nota sea entregada, encontrarás la resolución en este link de notion:

https://www.notion.so/R-SVM-Lab-Churn-5e7b5abf68f94dda96264ef41e1b5fae?source=copy_link

Recuerda que existen varias formar de solucionar las actividades propuestas en los laboratorios. Toma la resolución entregada como referencia. En caso de que el link de resolución no esté público, solicita acceso en el mismo link de notion.|

# Predicción de pérdida de clientes en Empresas de Servicios de Telecomunicaciones (Churn)

Reducir las salidas y [deserciones de clientes](https://www.bain.com/insights/breaking-the-back-of-customer-churn/) se ha convertido en una alta prioridad para la mayoría de los proveedores de servicios de comunicaciones a medida que los mercados maduran y la competencia se intensifica.




En este documento usaremos una base de datos de una empresa de telecomunicaciones anónima [disponibilizada por IBM](https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv).

El principal objetivo es crear un model de aprendizaje automático basado en SVM y Regresión Logística (Logistic Regression, LR) para predecir la pérdida o salida de clientes en una empresa de telecomunicaciones.


## Librerías

In [ ]:
# importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split


## Base de datos

Este conjunto de datos contiene un total de 7043 clientes y 21 características de los mismos. De las entradas, 5174 son clientes activos y 1869 son clientes que la empresa ha perdido. Observe que el conjunto de datos está desbalanceado pues por cada cliente perdido existe casi 3 clientes activos. La variable de salida para nuestro modelo de machine learning será `Churn`.

In [ ]:
# importamos dataset
DATA_PATH = "https://raw.githubusercontent.com/carlosfab/dsnp2/master/datasets/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(DATA_PATH)

# vemos las primeras 5 filas
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### *Detalles de la base de datos*

*	`customerID` - Custumer unique identifier
*	`gender` - Customer gender - ['Female' 'Male']
*	`SeniorCitizen` - Elderly or retired person, a senior citizen is someone who has at least attained the age of 60 of 65 years
*	`Partner` - - ['No' 'Yes']
*	`Dependents` - If customer has dependents - ['No' 'Yes']
*	`Tenure` - Customer lifespan (in months)
*	`PhoneService` - - ['No' 'Yes']
*	`MultipleLines` - - ['No' 'No phone service' 'Yes']
*	`InternetService` - - ['No' 'No internet service' 'Yes']
*	`OnlineSecurity` - - ['No' 'No internet service' 'Yes']
*	`OnlineBackup` - - ['No' 'No internet service' 'Yes']
*	`DeviceProtection` - - ['No' 'No internet service' 'Yes']
*	`TechSupport` - - ['No' 'No internet service' 'Yes']
*	`StreamingTV` - - ['No' 'No internet service' 'Yes']
*	`StreamingMovies` -  - ['No' 'No internet service' 'Yes']
*	`Contract` - Type of contract - ['Month-to-month' 'One year' 'Two year']
*	`PaperlessBilling` - - ['No' 'Yes']
*	`PaymentMethod` - payment method - ['Bank transfer (automatic)', 'Credit card (automatic)', 'Electronic check', 'Mailed check']
*	`MonthlyCharges` - Monthly Recurring Charges
*	`TotalCharges` - Life time value
*	`Churn` - Churn value, the targer vector - ['No' 'Yes']





### *Limpieza del Dataset*

La función get_df_size se utiliza para obtener información básica sobre el tamaño del conjunto de datos, en términos de su número de atributos y entradas.

Luego, se reemplazan los valores en blanco por NaN (valores faltantes) en el dataframe utilizando la función replace de pandas. Se reemplazan los valores faltantes en la columna TotalCharges por la mediana de los valores no faltantes de la misma columna, utilizando la función fillna de pandas. Además, se convierte esta columna de tipo objeto a tipo numérico utilizando la función apply de pandas y el método pd.to_numeric.

A continuación, se elimina la columna customerID, ya que no es una característica relevante para el modelo. Se utiliza el método drop de pandas para eliminar la columna, especificando el argumento axis=1 para indicar que se está eliminando una columna y no una fila.

Por último, se utiliza la función describe de pandas para obtener estadísticas descriptivas del conjunto de datos preprocesado y se imprimen en pantalla la cantidad de instancias con valor "Yes" y "No" en la variable objetivo Churn.

In [ ]:
def get_df_size(df, header='Dataset dimensions'):
  print(header,
        '\n# Attributes: ', df.shape[1],
        '\n# Entries: ', df.shape[0],'\n')

get_df_size(df)

#df.info()

# reemplaza valores en blanco por NaN
df_clean = df.replace(r'^\s*$', np.nan, regex=True)


# Convert 'TotalCharges' to numeric before calculating the median.
# errors='coerce' will handle any non-numeric values by setting them to NaN.
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')

# reemplaza valores faltantes en TotalCharges por la mediana de TotalCharges.
total_charges_median = df_clean.TotalCharges.median()


df_clean['TotalCharges'].fillna(total_charges_median, inplace=True)
df_clean['TotalCharges'] = df_clean['TotalCharges'].apply(pd.to_numeric)

#CustomerID lo retiramos porque no es una característica
df_clean = df_clean.drop('customerID', axis=1)
df_clean.describe()

print("Churn No Instances: ", df_clean[df_clean['Churn'] == 'No'].shape[0])
print("Churn Yes Instances: ", df_clean[df_clean['Churn'] == 'Yes'].shape[0])


Dataset dimensions 
# Attributes:  21 
# Entries:  7043 



<ipython-input-3-76449e648e48>:22: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean['TotalCharges'].fillna(total_charges_median, inplace=True)


Churn No Instances:  5174
Churn Yes Instances:  1869


## Preparación de la Base de Datos








Este código realiza la preprocesamiento de los datos en el dataframe df_clean para poder usarlo en un modelo de machine learning. En primer lugar, identifica las características binarias, numéricas y categóricas en el conjunto de datos. Luego, crea una copia del dataframe original y realiza las siguientes operaciones sobre él:

Convierte las características binarias en etiquetas numéricas mediante el uso de LabelEncoder().
Crea variables dummy para las características categóricas usando pd.get_dummies().
Imprime la información del tamaño del conjunto de datos original y procesado utilizando la función get_df_size().
Retorna un nuevo dataframe df_proc que contiene todas las características preprocesadas y que puede ser utilizado en un modelo de machine learning.
En resumen, el código realiza la limpieza, transformación y preparación de los datos para su posterior uso en un modelo de machine learning.

In [ ]:
binary_feat = df_clean.nunique()[df_clean.nunique() == 2].keys().tolist()
numeric_feat = [col for col in df_clean.select_dtypes(['float','int']).columns.tolist() if col not in binary_feat]
categorical_feat = [ col for col in df_clean.select_dtypes('object').columns.to_list() if col not in binary_feat + numeric_feat ]
df_proc = df_clean.copy()
#Etiquetas para características binarias
le = LabelEncoder()
for i in binary_feat:
  df_proc[i] = le.fit_transform(df_proc[i])
  print(i, '\n', np.unique(df_proc[i].values))
#Dummy variables
df_proc = pd.get_dummies(df_proc, columns=categorical_feat)
get_df_size(df, header='Original dataset:')
get_df_size(df_proc, header='Processed dataset:')
df_proc.head()

gender 
 [0 1]
SeniorCitizen 
 [0 1]
Partner 
 [0 1]
Dependents 
 [0 1]
PhoneService 
 [0 1]
PaperlessBilling 
 [0 1]
Churn 
 [0 1]
Original dataset: 
# Attributes:  21 
# Entries:  7043 

Processed dataset: 
# Attributes:  41 
# Entries:  7043 



,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,StreamingMovies_No,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_Month-to-month,Contract_One year,Contract_Two year,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,1,0,1,29.85,29.85,0,...,True,False,False,True,False,False,False,False,True,False
1,1,0,0,0,34,1,0,56.95,1889.50,0,...,True,False,False,False,True,False,False,False,False,True
2,1,0,0,0,2,1,1,53.85,108.15,1,...,True,False,False,True,False,False,False,False,False,True
3,1,0,0,0,45,0,0,42.30,1840.75,0,...,True,False,False,False,True,False,True,False,False,False
4,0,0,0,0,2,1,1,70.70,151.65,1,...,True,False,False,True,False,False,False,False,True,False


### *División en conjunto de entrenamiento y test*

In [ ]:
# dividimos df_proc en características y salida
X=df_proc.drop('Churn', axis=1)
y=df_proc['Churn']

# Dividimos el conjunto de entrenamiento y test
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

### ✅ Actividad 1: Visualización con TSNE
Utilice TSNE para visualizar su dataset de entrenamiento. Recuerde normalizar su dasaset (zscore) antes de esta y las siguientes actividades.

In [ ]:
# ---------------- Actividad 1: Visualización con TSNE ----------------
from sklearn.manifold import TSNE
import seaborn as sns

# Normalización z-score (se ajusta solo con el conjunto de entrenamiento)
scaler_tsne = StandardScaler()
X_train_z = scaler_tsne.fit_transform(X_train.astype(float))

tsne = TSNE(n_components=2, perplexity=30, init='pca', random_state=42)
X_train_tsne = tsne.fit_transform(X_train_z)

plt.figure(figsize=(9, 7))
for clase, color, nombre in [(0, 'tab:blue', 'Churn = No'), (1, 'tab:red', 'Churn = Yes')]:
    mask = (y_train.values == clase)
    plt.scatter(X_train_tsne[mask, 0], X_train_tsne[mask, 1], s=6, alpha=0.45, c=color, label=nombre)
plt.title('TSNE (2D) del conjunto de entrenamiento normalizado (z-score)')
plt.xlabel('TSNE 1'); plt.ylabel('TSNE 2')
plt.legend(markerscale=3)
plt.tight_layout()
plt.show()

**Observación (Actividad 1):** en la proyección TSNE los clientes que abandonan (rojo) no forman un grupo aislado: aparecen concentrados en algunas regiones (típicamente las de contrato mes a mes, fibra óptica y poca antigüedad) pero muy mezclados con clientes que se quedan. Esto anticipa que las clases **no son linealmente separables** y que habrá un compromiso entre recall y precisión: para detectar más clientes que se van, el modelo tendrá que aceptar falsos positivos. Además, el desbalance (≈ 26 % de churn) se ve claramente en la cantidad de puntos de cada color.

### ✅ Actividad 2: Clasificador y Optimización de Hiperparámetros SVM

Cree un clasificador con SVM (ver `SVC`) con kernel RBF.
Use 5-fold cross-validation (ver `GridSearchCV` y  `StratifiedKFold`) para optimizar los siguientes hiperparámetros en el conjunto de entrenamiento (`X_train`, `y_train`):

1. Gamma
2. C


Para la optimización de hiperparámetros utilice como métrica al recall.

Grafique con un mapa de calor 2D donde el eje x es Gamma, el eje y es C y el color es el score.

Una vez optimizados los hiperparámetros, reentrene su clasificador con los mejores hiperparámetros encontrados. Finalmente, evalúe este último clasificador en el conjunto de test ('X_test', 'y_test'.)




In [ ]:
# ---------------- Actividad 2: SVM (kernel RBF) + GridSearchCV ----------------
import time
import seaborn as sns
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import recall_score, f1_score, precision_score, ConfusionMatrixDisplay

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# El escalado va dentro del pipeline para que en cada fold se ajuste solo con los datos de entrenamiento.
# class_weight='balanced' compensa el desbalance de clases (≈ 3 a 1), clave cuando optimizamos recall.
pipe_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='rbf', class_weight='balanced', random_state=42))
])

param_grid_svm = {
    'svc__C':     [0.01, 0.1, 1, 10, 100],
    'svc__gamma': [1e-4, 1e-3, 1e-2, 1e-1, 1],
}

grid_svm = GridSearchCV(pipe_svm, param_grid_svm, scoring='recall', cv=kfold, n_jobs=-1, verbose=1)
t0 = time.time()
grid_svm.fit(X_train, y_train)
t_grid_svm = time.time() - t0

print(f"Tiempo GridSearchCV SVM: {t_grid_svm:.1f} s  ({len(grid_svm.cv_results_['params'])} combinaciones x 5 folds)")
print(f"Mejor recall CV: {grid_svm.best_score_:.4f} con {grid_svm.best_params_}")

# Mapa de calor: eje x = gamma, eje y = C, color = recall promedio de validación
res_svm = pd.DataFrame(grid_svm.cv_results_)
heat = res_svm.pivot_table(index='param_svc__C', columns='param_svc__gamma', values='mean_test_score')
plt.figure(figsize=(8, 6))
sns.heatmap(heat, annot=True, fmt='.3f', cmap='viridis', cbar_kws={'label': 'Recall CV (promedio 5 folds)'})
plt.xlabel('gamma'); plt.ylabel('C')
plt.title('SVM RBF – Recall de validación cruzada')
plt.tight_layout()
plt.show()

# Reentrenamiento con los mejores hiperparámetros sobre todo X_train
best_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='rbf', class_weight='balanced', random_state=42,
                C=grid_svm.best_params_['svc__C'], gamma=grid_svm.best_params_['svc__gamma']))
])
best_svm.fit(X_train, y_train)

pred_svm = best_svm.predict(X_test)
metrics_svm = {
    'recall':    recall_score(y_test, pred_svm),
    'f1':        f1_score(y_test, pred_svm),
    'accuracy':  accuracy_score(y_test, pred_svm),
    'precision': precision_score(y_test, pred_svm),
}
print("\nSVM optimizado – métricas en test:", {k: round(v, 4) for k, v in metrics_svm.items()})
print(classification_report(y_test, pred_svm, target_names=['No churn', 'Churn']))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred_svm), display_labels=['No churn', 'Churn']).plot(cmap='Blues')
plt.title('Matriz de confusión – SVM (test)')
plt.show()

**Análisis (Actividad 2):** el mapa de calor muestra cómo interactúan `C` y `gamma`. Al optimizar **recall** con `class_weight='balanced'`, las mejores celdas suelen estar en la zona de `gamma` pequeño/moderado (fronteras suaves): un `gamma` muy grande hace que cada vector de soporte tenga un radio de influencia mínimo, el modelo memoriza el entrenamiento y el recall de validación cae. Hay que tener en cuenta que el recall por sí solo puede premiar modelos que predicen "churn" con mucha frecuencia, por eso en test se reportan también precisión, F1 y la matriz de confusión: el modelo elegido detecta la mayoría de clientes que se van (pocos falsos negativos) a costa de un número apreciable de falsos positivos.

### ✅ Actividad 3: Clasificador y Optimización de Hiperparámetros Logistic Regression

Repita la actividad 2 pero con Logistic Regression. Bajo el mismo esquema de la actividad anterior, optimice el parámetro de regularización lambda. Utilice una transformación polinomial para garantizar fronteras de decisión no lineal. Compare sus resultados con los obtenidos de la actividad anterior

In [ ]:
#INSERTE AQUÍ SU CÓDIGO
# Recomendación de configuración para LR para aumentar la velocidad del entrenamiento
# Opción 1: LogisticRegression(max_iter=20000, solver="lbfgs", penalty="l2")
# Opción 2: Implemenatr LR como una red neuronal en pytorch

# ---------------- Actividad 3: Logistic Regression polinomial + GridSearchCV ----------------
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures

pipe_lr = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),   # fronteras de decisión no lineales
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(penalty='l2', solver='lbfgs', max_iter=20000, class_weight='balanced'))
])

# C = 1/lambda  ->  C pequeño = regularización fuerte
param_grid_lr = {'lr__C': np.logspace(-4, 2, 7)}

grid_lr = GridSearchCV(pipe_lr, param_grid_lr, scoring='recall', cv=kfold, n_jobs=-1, verbose=1)
t0 = time.time()
grid_lr.fit(X_train, y_train)
t_grid_lr = time.time() - t0

best_C = grid_lr.best_params_['lr__C']
print(f"Tiempo GridSearchCV LR: {t_grid_lr:.1f} s")
print(f"Mejor recall CV: {grid_lr.best_score_:.4f} con C = {best_C:g} (lambda = {1/best_C:g})")

# Curva recall vs C (equivale a un "heatmap" de un solo hiperparámetro)
res_lr = pd.DataFrame(grid_lr.cv_results_)
plt.figure(figsize=(8, 5))
plt.errorbar(res_lr['param_lr__C'].astype(float), res_lr['mean_test_score'], yerr=res_lr['std_test_score'],
             marker='o', capsize=4)
plt.xscale('log')
plt.xlabel('C (= 1/lambda, escala log)'); plt.ylabel('Recall CV (media ± std)')
plt.title('Logistic Regression polinomial (grado 2) – Recall vs C')
plt.grid(True, alpha=0.3)
plt.show()

# Reentrenamiento con el mejor C sobre todo X_train y evaluación en test
best_lr = pipe_lr.set_params(lr__C=best_C)
best_lr.fit(X_train, y_train)
pred_lr = best_lr.predict(X_test)
metrics_lr = {
    'recall':    recall_score(y_test, pred_lr),
    'f1':        f1_score(y_test, pred_lr),
    'accuracy':  accuracy_score(y_test, pred_lr),
    'precision': precision_score(y_test, pred_lr),
}
print(classification_report(y_test, pred_lr, target_names=['No churn', 'Churn']))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred_lr), display_labels=['No churn', 'Churn']).plot(cmap='Greens')
plt.title('Matriz de confusión – LR polinomial (test)')
plt.show()

# Comparación SVM vs LR
comparacion = pd.DataFrame({'SVM RBF': metrics_svm, 'LR polinomial': metrics_lr}).T.round(4)
comparacion['tiempo GridSearch (s)'] = [round(t_grid_svm, 1), round(t_grid_lr, 1)]
display(comparacion)
for m in ['recall', 'f1', 'accuracy', 'precision']:
    ganador = comparacion[m].idxmax()
    print(f"Mejor en {m:9s}: {ganador}  ({comparacion.loc[ganador, m]:.4f})")

**Comparación SVM vs Logistic Regression (Actividad 3):**

- Ambos modelos usan el mismo esquema (z-score, 5-fold estratificado, `scoring='recall'`, `class_weight='balanced'`), así que la tabla anterior es una comparación justa.
- La **LR polinomial de grado 2** genera ~800 características (todas las interacciones de pares); la regularización L2 (`C`) es la que controla que no sobreajuste: con `C` muy pequeño el modelo se vuelve casi constante y con `C` grande se sobreajusta a las interacciones, por eso la curva recall-vs-C tiene un máximo intermedio.
- El **SVM RBF** obtiene fronteras no lineales sin expandir explícitamente las características (truco del kernel), pero tiene dos hiperparámetros y su entrenamiento escala peor con el número de muestras (el tiempo de GridSearch suele ser mayor).
- **Conclusión:** para churn la métrica prioritaria es el **recall** (un cliente que se va y no se detecta es una pérdida directa de ingresos, mientras que un falso positivo solo cuesta una acción de retención). Se prefiere el modelo con mayor recall en test siempre que su precisión no se desplome; si los recalls son similares (diferencias de 1–2 puntos), la **LR** es preferible por ser más rápida, más interpretable (coeficientes) y entregar probabilidades calibradas que permiten mover el umbral de decisión según el presupuesto de retención.

### ✅ Actividad 4: Bayesian Optimization

Repetir la actividad 2 usando Bayesian Optimization

Tip: usar BayesSearchCV

https://machinelearningmastery.com/scikit-optimize-for-hyperparameter-tuning-in-machine-learning/


In [ ]:
# ---------------- Actividad 4: Bayesian Optimization (BayesSearchCV) para SVM RBF ----------------
!pip install scikit-optimize --quiet
from skopt import BayesSearchCV
from skopt.space import Real

search_space_svm = {
    'svc__C':     Real(1e-2, 1e2, prior='log-uniform'),
    'svc__gamma': Real(1e-4, 1e0, prior='log-uniform'),
}

bayes_svm = BayesSearchCV(
    Pipeline([('scaler', StandardScaler()),
              ('svc', SVC(kernel='rbf', class_weight='balanced', random_state=42))]),
    search_spaces=search_space_svm,
    n_iter=25,                 # mismo orden de evaluaciones que la malla 5x5 del GridSearch
    scoring='recall',
    cv=kfold,
    n_jobs=-1,
    random_state=42,
)
t0 = time.time()
bayes_svm.fit(X_train, y_train)
t_bayes_svm = time.time() - t0

print(f"Mejor recall CV (Bayes): {bayes_svm.best_score_:.4f} con {dict(bayes_svm.best_params_)}")

pred_bayes = bayes_svm.best_estimator_.predict(X_test)
metrics_bayes = {
    'recall':    recall_score(y_test, pred_bayes),
    'f1':        f1_score(y_test, pred_bayes),
    'accuracy':  accuracy_score(y_test, pred_bayes),
    'precision': precision_score(y_test, pred_bayes),
}

resumen = pd.DataFrame({
    'Grid Search': {'mejor recall CV': grid_svm.best_score_, 'tiempo (s)': t_grid_svm,
                    'evaluaciones': len(grid_svm.cv_results_['params']), **{f'{k} test': v for k, v in metrics_svm.items()}},
    'Bayes Search': {'mejor recall CV': bayes_svm.best_score_, 'tiempo (s)': t_bayes_svm,
                     'evaluaciones': len(bayes_svm.cv_results_['params']), **{f'{k} test': v for k, v in metrics_bayes.items()}},
}).T.round(4)
display(resumen)

# Trayectoria de la búsqueda bayesiana: mejor recall acumulado vs iteración
scores_iter = np.array(bayes_svm.cv_results_['mean_test_score'])
plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, len(scores_iter) + 1), np.maximum.accumulate(scores_iter), 'o-', label='Bayes: mejor recall acumulado')
plt.axhline(grid_svm.best_score_, color='red', ls='--', label='Mejor recall Grid Search')
plt.xlabel('Iteración'); plt.ylabel('Recall CV'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Convergencia de BayesSearchCV')
plt.show()

**Grid Search vs Bayesian Optimization (Actividad 4):** con el mismo presupuesto aproximado de evaluaciones (25), la búsqueda bayesiana explora un espacio **continuo** de `C` y `gamma` (no solo los 5 valores fijos por eje de la malla) y concentra las evaluaciones en la región prometedora que el modelo sustituto (proceso gaussiano) va aprendiendo; por eso alcanza un recall de validación igual o ligeramente superior al del Grid Search, normalmente en las primeras iteraciones (ver la curva de convergencia). El tiempo total es similar o un poco mayor porque las evaluaciones son secuenciales (cada punto depende de los anteriores) y el ajuste del proceso gaussiano añade overhead; la ventaja real aparece cuando el espacio de hiperparámetros crece, ya que el costo del grid crece exponencialmente con el número de hiperparámetros mientras que la búsqueda bayesiana mantiene un número fijo de evaluaciones.